## Importing data and libraries

In [2]:
import pandas as pd 
import numpy as np
import optuna
import xgboost as xgb 
import lightgbm as lgb
import catboost as cb
from optbinning import OptimalBinning
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import plotly.graph_objects as go

c:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
(CVXPY) May 24 12:20:48 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) May 24 12:20:48 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


In [3]:
AUC_BENCHMARK = 0.95104

print("Reading data ...")
train_org = pd.read_csv("playground-series-s6e5\\train.csv")
test_org = pd.read_csv("playground-series-s6e5\\test.csv")
submissions_org = pd.read_csv("playground-series-s6e5\\sample_submission.csv")

train_org.shape, test_org.shape, submissions_org.shape

Reading data ...


((439140, 16), (188165, 15), (188165, 2))

## Configurations
- Binning
- Feature Engineering

In [35]:
F_ENGG = True
MAX_BIN = 25
MIN_BIN = 0.06
SHOW_WOE = False
OPTUNA = True

## Research & Feature Engineering

In [25]:
def engineer_race_features(df, activate=True):

    if not activate:
        return df

    df = df.copy()

    eps = 1e-5

    # =========================================================
    # BASIC PROGRESS FEATURES
    # =========================================================

    df['Progress_Per_Lap_engg'] = (
        df['RaceProgress'] / (df['LapNumber'] + eps)
    )

    df['Remaining_RaceProgress_engg'] = (
        1 - df['RaceProgress']
    )

    df['Remaining_Laps_Ratio_engg'] = (
        (1 - df['RaceProgress']) /
        (df['LapNumber'] + eps)
    )

    # =========================================================
    # DEGRADATION FEATURES
    # =========================================================

    df['Deg_Per_Lap_engg'] = (
        df['Cumulative_Degradation'] /
        (df['LapNumber'] + eps)
    )

    df['Deg_Per_TyreLife_engg'] = (
        df['Cumulative_Degradation'] /
        (df['TyreLife'] + eps)
    )

    df['Deg_x_TyreLife_engg'] = (
        df['Cumulative_Degradation'] *
        df['TyreLife']
    )

    df['Deg_x_Progress_engg'] = (
        df['Cumulative_Degradation'] *
        df['RaceProgress']
    )

    df['Deg_Acceleration_engg'] = (
        df['Cumulative_Degradation'] /
        (df['RaceProgress'] + eps)
    )

    # =========================================================
    # PACE FEATURES
    # =========================================================

    df['Pace_Tyre_Sensitivity_engg'] = (
        df['LapTime_Delta'] /
        (df['TyreLife'] + eps)
    )

    df['Pace_Per_Position_engg'] = (
        df['LapTime_Delta'] /
        (df['Position'] + eps)
    )

    df['LapTime_x_TyreLife_engg'] = (
        df['LapTime_Delta'] *
        df['TyreLife']
    )

    df['LapTime_x_Deg_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['LapTime_x_Progress_engg'] = (
        df['LapTime_Delta'] *
        df['RaceProgress']
    )

    df['Pace_Drop_Flag_engg'] = (
        df['LapTime_Delta'] > 0
    ).astype(int)

    df['Extreme_Pace_Drop_engg'] = (
        df['LapTime_Delta'] > df['LapTime_Delta'].quantile(0.90)
    ).astype(int)

    # =========================================================
    # TYRE FEATURES
    # =========================================================

    df['TyreLife_Per_Lap_engg'] = (
        df['TyreLife'] /
        (df['LapNumber'] + eps)
    )

    df['TyreLife_Per_Progress_engg'] = (
        df['TyreLife'] /
        (df['RaceProgress'] + eps)
    )

    df['TyreLife_x_Progress_engg'] = (
        df['TyreLife'] *
        df['RaceProgress']
    )

    df['Fresh_Tyre_Flag_engg'] = (
        df['TyreLife'] <= 5
    ).astype(int)

    df['Medium_Tyre_Flag_engg'] = (
        (df['TyreLife'] > 5) &
        (df['TyreLife'] <= 20)
    ).astype(int)

    df['Old_Tyre_Flag_engg'] = (
        df['TyreLife'] > 20
    ).astype(int)

    # =========================================================
    # POSITION FEATURES
    # =========================================================

    df['Losing_Ground_engg'] = (
        df['Position_Change'] < 0
    ).astype(int)

    df['Gaining_Ground_engg'] = (
        df['Position_Change'] > 0
    ).astype(int)

    df['Position_x_Progress_engg'] = (
        df['Position'] *
        df['RaceProgress']
    )

    df['Position_x_TyreLife_engg'] = (
        df['Position'] *
        df['TyreLife']
    )

    df['Position_Change_Intensity_engg'] = (
        df['Position_Change'] /
        (df['LapNumber'] + eps)
    )

    df['Bad_Position_Flag_engg'] = (
        df['Position'] > 10
    ).astype(int)

    df['Podium_Position_Flag_engg'] = (
        df['Position'] <= 3
    ).astype(int)

    # =========================================================
    # PIT WINDOW FEATURES
    # =========================================================

    df['Potential_Pit_Window_engg'] = (
        (df['TyreLife'] > 15) &
        (df['RaceProgress'] > 0.25)
    ).astype(int)

    df['Late_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] > 0.70)
    ).astype(int)

    df['Early_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] < 0.25)
    ).astype(int)

    # =========================================================
    # INTERACTION FEATURES
    # =========================================================

    df['Wear_Pace_Impact_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['Wear_Position_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position']
    )

    df['Wear_Position_Change_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position_Change']
    )

    df['TyreLife_Position_Interaction_engg'] = (
        df['TyreLife'] *
        df['Position']
    )

    df['TyreLife_Lap_Interaction_engg'] = (
        df['TyreLife'] *
        df['LapNumber']
    )

    df['TyreLife_Stint_Interaction_engg'] = (
        df['TyreLife'] *
        df['Stint']
    )

    df['Lap_Position_Interaction_engg'] = (
        df['LapNumber'] *
        df['Position']
    )

    # =========================================================
    # STINT FEATURES
    # =========================================================

    df['Is_First_Stint_engg'] = (
        df['Stint'] == 1
    ).astype(int)

    df['Is_Second_Stint_engg'] = (
        df['Stint'] == 2
    ).astype(int)

    df['Is_ThirdPlus_Stint_engg'] = (
        df['Stint'] >= 3
    ).astype(int)

    df['Stint_x_Progress_engg'] = (
        df['Stint'] *
        df['RaceProgress']
    )

    df['Stint_x_TyreLife_engg'] = (
        df['Stint'] *
        df['TyreLife']
    )

    # =========================================================
    # CATEGORICAL COMBINATIONS
    # =========================================================

    df['Year_Stint_engg'] = (
        df['Year'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Year_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Race_Stint_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Driver_Compound_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Driver_Race_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Race'].astype(str)
    )

    df['Compound_Stint_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Position_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Position'].astype(str)
    )

    df['Race_Compound_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Race_Year_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Driver_Stint_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    return df
    

train_df = engineer_race_features(train_org, activate=F_ENGG)
test_df = engineer_race_features(test_org, activate=F_ENGG)

t_col = "PitNextLap"
ID = "id"
c_cols = list(test_df.select_dtypes(include='object').columns)
n_cols = list(test_df.select_dtypes(exclude='object').columns)

print("--" * 20, "Categorical Columns", "--" * 20)
print(train_df[c_cols].head())
print("--" * 20, "Numerical Columns", "--" * 20)
print(train_df[n_cols].head())

---------------------------------------- Categorical Columns ----------------------------------------
  Driver Compound                   Race Year_Stint_engg Compound_Year_engg  \
0   D109     HARD    Canadian Grand Prix          2022_2          HARD_2022   
1   D086     HARD       Dutch Grand Prix          2025_2          HARD_2025   
2    ZON     HARD    Austrian Grand Prix          2022_3          HARD_2022   
3    SPE   MEDIUM     Pre-Season Testing          2023_1        MEDIUM_2023   
4   D019     HARD  Azerbaijan Grand Prix          2022_3          HARD_2022   

           Race_Stint_engg Driver_Compound_engg            Driver_Race_engg  \
0    Canadian Grand Prix_2            D109_HARD    D109_Canadian Grand Prix   
1       Dutch Grand Prix_2            D086_HARD       D086_Dutch Grand Prix   
2    Austrian Grand Prix_3             ZON_HARD     ZON_Austrian Grand Prix   
3     Pre-Season Testing_1           SPE_MEDIUM      SPE_Pre-Season Testing   
4  Azerbaijan Grand Prix_3  

C:\Users\admin\AppData\Local\Temp\ipykernel_11700\2930407169.py:314: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  c_cols = list(test_df.select_dtypes(include='object').columns)


## Optbinning / WOE / IV
- Categorical binning
- Numerical binning
- Top features by IV
- WOE Trend for each feature

In [30]:
class OptimalBinner:

    def __init__(
        self, cat_cols: list, num_cols: list, max_n_bins: int = 5, min_bin_size: float = 0.05):
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        self.max_n_bins = max_n_bins
        self.min_bin_size = min_bin_size
        self.binners = {}

    def fit(self, X, y, verbos=True):
        for col in X.columns:
            if verbos:
                print(f"Fitting: {col}")
            if col in self.cat_cols:
                optb = OptimalBinning(
                    name=col,
                    dtype="categorical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )

            else:
                optb = OptimalBinning(
                    name=col,
                    dtype="numerical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )
            # Fit binning
            optb.fit(X[col], y)
            self.binners[col] = optb
        return self

    def transform(self, X, metric="woe"):
        X_transformed = pd.DataFrame(index=X.index)
        for col in X.columns:
            optb = self.binners[col]
            X_transformed[col] = optb.transform(
                X[col],
                metric=metric,
                metric_missing="empirical",
                metric_special="empirical"
            )
        return X_transformed

    def get_iv_summary(self):
        iv_data = []
        for col, optb in self.binners.items():
            iv = optb.binning_table.build()["IV"].sum()
            iv_data.append({
                "feature": col,
                "iv": iv
            })
        return (
            pd.DataFrame(iv_data)
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

    def get_binning_table(self, col):
        return self.binners[col].binning_table.build()


class LGB_XGB_CAT_Model:

    def __init__(self, lgb_params=None, xgb_params=None, cb_params=None):

        self.lgb_model = None
        self.xgb_model = None
        self.cb_model = None

        self.lgb_params = lgb_params
        self.xgb_params = xgb_params
        self.cb_params = cb_params

    def train(self, X_train, y_train):

        if type(self.lgb_params) != type(None):
            print("Training LGB model...")
            self.lgb_model = lgb.LGBMClassifier(
                **self.lgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.xgb_params) != type(None):
            print("Training XGB model...")
            self.xgb_model = xgb.XGBClassifier(
                **self.xgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.cb_params) != type(None):
            print("Training CatBoost model...")
            self.cb_model = cb.CatBoostClassifier(
                **self.cb_params
            ).fit(
                X_train,
                y_train
            )
        print("Models trained successfully!")

    def predict_proba(self, X):
        probs = {}
        if type(self.lgb_params) != type(None):
            probs['lgb'] = (
                self.lgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.xgb_params) != type(None):
            probs['xgb'] = (
                self.xgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.cb_params) != type(None):
            probs['cb'] = (
                self.cb_model
                .predict_proba(X)[:, 1]
            )
        return probs

    def roc_auc_score(self, X_test, y_test):
        roc_auc_dict = {}
        if type(self.lgb_params) != type(None):
            roc_auc_dict['lgb'] = roc_auc_score(y_test, self.lgb_model.predict_proba(X_test)[:, 1])

        if type(self.xgb_params) != type(None):
            roc_auc_dict['xgb'] = roc_auc_score(y_test, self.xgb_model.predict_proba(X_test)[:, 1])

        if type(self.cb_params) != type(None):
            roc_auc_dict['cb'] = roc_auc_score(y_test, self.cb_model.predict_proba(X_test)[:, 1])

        return roc_auc_dict

    def cross_validation(self, X, y, groups, n_splits=5):
        gkf = GroupKFold(
            n_splits=n_splits
        )
        cv_scores = {
            "lgb": [],
            "xgb": [],
            "cb": []
        }

        for train_idx, valid_idx in gkf.split(X, y, groups=groups):
            X_train_cv = X.iloc[train_idx]
            X_valid_cv = X.iloc[valid_idx]

            y_train_cv = y.iloc[train_idx]
            y_valid_cv = y.iloc[valid_idx]

            if type(self.lgb_params) != type(None):

                model_lgb = lgb.LGBMClassifier(**self.lgb_params)
                model_lgb.fit(X_train_cv, y_train_cv)
                y_prob_lgb = model_lgb.predict_proba(X_valid_cv)[:, 1]
                auc_lgb = roc_auc_score(y_valid_cv, y_prob_lgb)
                cv_scores["lgb"].append(auc_lgb)

            if type(self.xgb_params) != type(None):
                model_xgb = xgb.XGBClassifier(**self.xgb_params)
                model_xgb.fit(X_train_cv, y_train_cv)
                y_prob_xgb = model_xgb.predict_proba(X_valid_cv)[:, 1]
                auc_xgb = roc_auc_score(y_valid_cv, y_prob_xgb)
                cv_scores["xgb"].append(auc_xgb)

            if type(self.cb_params) != type(None):
                model_cb = cb.CatBoostClassifier(**self.cb_params)
                model_cb.fit(X_train_cv, y_train_cv, verbose=False)
                y_prob_cb = model_cb.predict_proba(X_valid_cv)[:, 1]
                auc_cb = roc_auc_score(y_valid_cv, y_prob_cb)
                cv_scores["cb"].append(auc_cb)

        final_scores = {}

        if len(cv_scores["lgb"]) > 0:
            final_scores["lgb"] = np.mean(cv_scores["lgb"])

        if len(cv_scores["xgb"]) > 0:
            final_scores["xgb"] = np.mean(cv_scores["xgb"])

        if len(cv_scores["cb"]) > 0:
            final_scores["cb"] = np.mean(cv_scores["cb"])

        return final_scores


X = train_df.drop(columns=[t_col, ID], axis='columns')
y = train_df[t_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,  stratify=y)
oot_df = test_df.drop(columns=[ID], axis='columns').copy()


if MAX_BIN == None and MIN_BIN == None:
    max_n_bins = [5, 10, 15, 20, 25]
    min_bin_size = [0.001, 0.05, 0.06, 0.07]
    best_bin_range = {}
    best_roc_auc = 0.0

    print(f"Searching for best max_bins and min_bins combination...")
    for max_bin in max_n_bins:
        for min_bin in min_bin_size:
            print(f"Fitting with max_bin = {max_bin} and min_bin = {min_bin}...")
            bin_obj = OptimalBinner(
                cat_cols=c_cols,
                num_cols=n_cols,
                max_n_bins=max_bin,
                min_bin_size=min_bin
            )

            bin_obj.fit(X_train, y_train, verbos=False)
            X_test_woe = bin_obj.transform(X_test, metric="woe")

            X_test_train, X_test_test, y_test_train, y_test_test = train_test_split(X_test_woe, y_test, test_size=0.3, random_state=42,  stratify=y_test)
            
            model_base = LGB_XGB_CAT_Model(xgb_params={})
            model_base.train(X_test_train, y_test_train)
            
            auc = model_base.roc_auc_score(X_test_test, y_test_test).values()
            avg_auc = sum(list(auc)) / len(auc)
            
            if best_roc_auc < avg_auc:
                best_roc_auc = avg_auc
                best_bin_range['max_bin'] = max_bin
                best_bin_range['min_bin'] = min_bin
            print(f"BIN RANGE : {max_bin, min_bin} | BEST BIN RANGE : {best_bin_range} | AUC : {avg_auc} | BEST AUC : {best_roc_auc}")

    MAX_BIN = best_bin_range['max_bin']
    MIN_BIN = best_bin_range['min_bin']

bin_obj = OptimalBinner(
    cat_cols=c_cols,
    num_cols=n_cols,
    max_n_bins=MAX_BIN,
    min_bin_size=MIN_BIN
)

bin_obj.fit(X_train, y_train, verbos=False)

X_train_woe = bin_obj.transform(X_train, metric="woe")
X_test_woe = bin_obj.transform(X_test, metric="woe")

print(X_train_woe.shape)
print(X_test_woe.shape)

Searching for best max_bins and min_bins combination...
Fitting with max_bin = 5 and min_bin = 0.001...
Training XGB model...
Models trained successfully!
BIN RANGE : (5, 0.001) | BEST BIN RANGE : {'max_bin': 5, 'min_bin': 0.001} | AUC : 0.9403408032898386 | BEST AUC : 0.9403408032898386
Fitting with max_bin = 5 and min_bin = 0.05...
Training XGB model...
Models trained successfully!
BIN RANGE : (5, 0.05) | BEST BIN RANGE : {'max_bin': 5, 'min_bin': 0.001} | AUC : 0.9403408032898386 | BEST AUC : 0.9403408032898386
Fitting with max_bin = 5 and min_bin = 0.06...
Training XGB model...
Models trained successfully!
BIN RANGE : (5, 0.06) | BEST BIN RANGE : {'max_bin': 5, 'min_bin': 0.06} | AUC : 0.9408666212822249 | BEST AUC : 0.9408666212822249
Fitting with max_bin = 5 and min_bin = 0.07...
Training XGB model...
Models trained successfully!
BIN RANGE : (5, 0.07) | BEST BIN RANGE : {'max_bin': 5, 'min_bin': 0.06} | AUC : 0.9402144729776682 | BEST AUC : 0.9408666212822249
Fitting with max_bin

In [36]:
iv_df = bin_obj.get_iv_summary()

top_iv = (
    iv_df
    .sort_values("iv", ascending=True)
    .reset_index(drop=True)
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_iv["iv"],
        y=top_iv["feature"],
        orientation="h",
        text=top_iv["iv"].round(3),
        textposition="outside"
    )
)

fig.update_layout(
    title="Interactive Information Value (IV) Analysis",
    xaxis_title="Information Value",
    yaxis_title="Feature",
    template="plotly_white",
    height=max(500, len(top_iv) * 35),
    hovermode="closest",
    showlegend=False
)

fig.show()

## WOE Trends

In [37]:
def plot_woe_trend(
    bin_obj,
    feature_name,
    figsize_height=500
    ):

    bt = (
        bin_obj
        .binners[feature_name]
        .binning_table
        .build()
    )

    bt = bt[
        ~bt["Bin"].astype(str).isin(
            ["Totals", "Special", "Missing"]
        )
    ].copy()


    fig = go.Figure()

    fig.add_trace(

        go.Scatter(
            x=bt["Bin"].astype(str),
            y=bt["WoE"],

            mode="lines+markers",

            name="WOE",

            yaxis="y1",

            hovertemplate=
            "<b>Bin:</b> %{x}<br>" +
            "<b>WOE:</b> %{y:.4f}<extra></extra>"
        )
    )


    fig.add_trace(

        go.Bar(
            x=bt["Bin"].astype(str),
            y=bt["Event rate"],

            name="Event Rate",

            yaxis="y2",

            opacity=0.5,

            hovertemplate=
            "<b>Bin:</b> %{x}<br>" +
            "<b>Event Rate:</b> %{y:.4f}<extra></extra>"
        )
    )

    fig.update_layout(

        title=f"WOE Trend Analysis: {feature_name}",

        xaxis=dict(
            title="Bins"
        ),

        yaxis=dict(
            title="WOE",
            side="left"
        ),

        yaxis2=dict(
            title="Event Rate",
            overlaying="y",
            side="right"
        ),

        template="plotly_white",

        hovermode="x unified",

        height=figsize_height,

        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.show()

if SHOW_WOE:
    for feature in X_train_woe.columns:
        if feature.lower() == 'driver':
            continue
        plot_woe_trend(
            bin_obj,
            feature_name=feature
        )

## Optuna Search
- LightGBM
- XGBoost

In [ ]:
SPW = len(y[y == 0]) / len(y[y == 1])

def objective(trial):

    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 15
        ),
        "gamma": trial.suggest_float(
            "gamma", 0, 10
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-5, 10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-5, 10,
            log=True
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.1,
            log=True
        ),
        "n_estimators": trial.suggest_int(
            "n_estimators", 300, 3000
        ),
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": SPW
    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    auc_scores = []

    for train_idx, valid_idx in cv.split(X_test_woe, y_test):

        X_train_cv = X_test_woe.iloc[train_idx]
        X_valid_cv = X_test_woe.iloc[valid_idx]

        y_train_cv = y_test.iloc[train_idx]
        y_valid_cv = y_test.iloc[valid_idx]

        model = xgb.XGBClassifier(
            **params
        )

        model.fit(
            X_train_cv,
            y_train_cv,

            eval_set=[
                (X_valid_cv, y_valid_cv)
            ],

            verbose=False
        )
        y_prob = model.predict_proba(
            X_valid_cv
        )[:, 1]

        auc = roc_auc_score(
            y_valid_cv,
            y_prob
        )
        auc_scores.append(auc)
    return np.mean(auc_scores)


if OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        study_name="xgb_roc_auc"
    )

    study.optimize(
        objective,
        n_trials=50,
        show_progress_bar=True
    )

    print("Best ROC AUC:", study.best_value)
    print("\nBest Parameters:\n")
    for k, v in study.best_params.items():
        print(f"{k}: {v}")

[I 2026-05-24 01:08:32,018] A new study created in memory with name: xgb_roc_auc
  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.943095:   2%|▏         | 1/50 [01:57<1:36:18, 117.93s/it]

[I 2026-05-24 01:10:29,937] Trial 0 finished with value: 0.9430947755132225 and parameters: {'max_depth': 8, 'min_child_weight': 12, 'gamma': 8.557507303844439, 'subsample': 0.9919965574668295, 'colsample_bytree': 0.8588053698248768, 'reg_alpha': 0.0012160967704232045, 'reg_lambda': 3.1693218343009737, 'learning_rate': 0.051070368284123945, 'n_estimators': 1370}. Best is trial 0 with value: 0.9430947755132225.


Best trial: 0. Best value: 0.943095:   4%|▍         | 2/50 [04:41<1:55:36, 144.50s/it]

[I 2026-05-24 01:13:13,040] Trial 1 finished with value: 0.9425165129806924 and parameters: {'max_depth': 5, 'min_child_weight': 1, 'gamma': 7.987845532990326, 'subsample': 0.8713479657900871, 'colsample_bytree': 0.914693721954523, 'reg_alpha': 0.1043866431085996, 'reg_lambda': 4.653557018943137e-05, 'learning_rate': 0.020080699521217292, 'n_estimators': 1556}. Best is trial 0 with value: 0.9430947755132225.


In [ ]:
import json

with open("xgb_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)

print("\nBest parameters saved to xgb_params.json")